# CS204 — Heart: бинарная классификация

Готовый ноутбук для задания по Keras/TensorFlow на наборе данных о сердечных заболеваниях.

Модель:
- 29 входных признаков;
- один скрытый слой из 16 нейронов с `ReLU`;
- один выходной нейрон с `sigmoid`;
- оптимизатор `Adam`;
- функция потерь `binary_crossentropy`.

Ноутбук рассчитан на запуск из папки, где находится файл `heart.csv`.


## 1. Импорт библиотек

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt


## 2. Фиксация случайности

Фиксируем seed, чтобы результаты запуска были воспроизводимыми.


In [ ]:
keras.utils.set_random_seed(42)


## 3. Загрузка данных

In [ ]:
df = pd.read_csv("heart.csv")


In [ ]:
df.shape


In [ ]:
df.head(3)


In [ ]:
df.target.value_counts(normalize=True, dropna=False)


## 4. Предварительная обработка

Категориальные признаки кодируем методом one-hot encoding.
Числовые признаки позднее стандартизируем по среднему и стандартному отклонению, рассчитанным только на обучающей выборке.


In [ ]:
categorical_variables = [
    'sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal'
]

numerics = [
    'age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'slope'
]


In [ ]:
df1 = pd.get_dummies(
    df,
    columns=categorical_variables,
    dtype=int
)

df1.head(3)


## 5. Разделение на обучающую и тестовую выборки

Используем 80% данных для обучения и 20% для тестирования.


In [ ]:
test_df = df1.sample(
    frac=0.2,
    random_state=42
)

train_df = df1.drop(test_df.index)


In [ ]:
train_df.shape


In [ ]:
test_df.shape


## 6. Нормализация числовых признаков

Среднее и стандартное отклонение вычисляются только по обучающей выборке, чтобы информация из тестовой выборки не попала в процесс обучения.


In [ ]:
means = train_df[numerics].mean()
sd = train_df[numerics].std()


In [ ]:
means


In [ ]:
train_df[numerics] = (
    train_df[numerics] - means
) / sd

test_df[numerics] = (
    test_df[numerics] - means
) / sd


In [ ]:
train_df.head()


## 7. Подготовка массивов NumPy и разделение X/y

In [ ]:
train = train_df.to_numpy()
test = test_df.to_numpy()


In [ ]:
# После one-hot encoding target находится в столбце с индексом 6.
train_X = np.delete(train, 6, axis=1)
test_X = np.delete(test, 6, axis=1)

train_X.shape, test_X.shape


In [ ]:
train_y = train[:, 6]
test_y = test[:, 6]

train_y.shape, test_y.shape


## 8. Создание модели Keras

In [ ]:
num_columns = train_X.shape[1]

# Входной слой
input = keras.Input(shape=(num_columns,))

# Скрытый слой: 16 нейронов, ReLU
h = keras.layers.Dense(
    16,
    activation="relu",
    name="Hidden"
)(input)

# Выходной слой: бинарная классификация
output = keras.layers.Dense(
    1,
    activation="sigmoid",
    name="Output"
)(h)

model = keras.Model(input, output)


In [ ]:
model.summary()


In [ ]:
# Проверка количества обучаемых параметров
(29 + 1) * 16 + (16 + 1) * 1


In [ ]:
keras.utils.plot_model(
    model,
    show_shapes=True
)


## 9. Компиляция модели

Для бинарной классификации используем `binary_crossentropy`, оптимизатор `Adam` и метрику `accuracy`.


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


## 10. Обучение модели

Обучаем модель 300 эпох с размером пакета 32.
20% обучающих данных автоматически используются как validation set.


In [ ]:
history = model.fit(
    train_X,
    train_y,
    epochs=300,
    batch_size=32,
    verbose=1,
    validation_split=0.2
)


## 11. Анализ обучения

In [ ]:
history_dict = history.history
history_dict.keys()


In [ ]:
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]

epochs = range(1, len(loss_values) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]

plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


## 12. Оценка модели на тестовой выборке

Финальная оценка выполняется на данных, которые модель не использовала при обучении.


In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_X,
    test_y,
    verbose=1
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


## 13. Вывод

Модель решает задачу бинарной классификации: по 29 обработанным признакам пациента она предсказывает вероятность наличия заболевания.

При запуске ноутбука итоговая точность на тестовой выборке будет выведена в предыдущей ячейке.
